In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
mol.precision = 1e-10
mol.build()
mf = scf.RHF(mol).density_fit()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-23-generic', version='#23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Tue May  5 21:11:26 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [3]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.40609899
CCSD Corr = -0.42461940
CCSD(T) Corr = -0.43105819


In [27]:
mol.aoslice_by_atom()

array([[ 0,  5,  0, 14],
       [ 5,  8, 14, 19],
       [ 8, 11, 19, 24],
       [11, 16, 24, 38],
       [16, 19, 38, 43],
       [19, 22, 43, 48]])

In [ ]:
aoslices = mol.aoslice_by_atom()

for ia in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[ia]
    symb = mol.atom_symbol(ia)        # e.g. 'O', 'H1', 'H@2' (keeps any user labels)
    # elem = mol.atom_pure_symbol(ia)   # e.g. 'O', 'H'        (just the element)
    # Z    = mol.atom_charge(ia)        # nuclear charge
    # xyz  = mol.atom_coord(ia)         # coordinates in Bohr
    print(f"Atom {ia} ({symb})")

Atom 0 (O) O
Atom 1 (H) H
Atom 2 (H) H
Atom 3 (O) O
Atom 4 (H) H
Atom 5 (H) H


In [18]:
from pyscf.lno import LNOCCSD, LNOCCSD_T
from pyscf.lno.tools import autofrag_iao

# def test_lno_iao_by_thresh(self):
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_coeff = lo.iao.iao(mol, orbocc)
lo_coeff = lo.orth.vec_lowdin(lo_coeff, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

frag_lolist = autofrag_iao(moliao)

gamma = 10
threshs = [3e-5]
# refs = [
#     [-0.4054784012,-0.4240686326,-0.4303996712],
#     [-0.4060479828,-0.4245745223,-0.4309965749],
# ]

emp2_iao = np.zeros(len(threshs))
eccsd_iao = np.zeros(len(threshs))
eccsd_t_iao = np.zeros(len(threshs))

for i, thresh in enumerate(threshs):
    mcc = LNOCCSD(mf, lo_coeff, frag_lolist, frozen=frozen).set(verbose=5)
    mcc.lno_thresh = [thresh*gamma,thresh]
    mcc.kernel()
    emp2_iao[i] = mcc.e_corr_pt2
    eccsd_iao[i] = mcc.e_corr_ccsd
    eccsd_t_iao[i] = mcc.e_corr_ccsd_t


******** <class 'pyscf.lno.lnoccsd.LNOCCSD'> ********
nocc = 8, nmo = 46
frozen orbitals 2
max_memory 4000 MB (current use 371 MB)
nfrag = 6  nlo = 14
frag_lolist = [[0, 1, 2, 3, 4], [5], [6], [7, 8, 9, 10, 11], [12], [13]]
frag_wghtlist = None
lno_type = ['1h', '1h']
lno_thresh = [0.00030000000000000003, 3e-05]
lno_pct_occ = None
lno_norb = None
lo_proj_thresh = 1e-10
lo_proj_thresh_active = 0.1
verbose_imp = 2
_ovL = None
_ovL_to_save = None
force_outcore_ao2mo = False
_match_oldcode = False
_max_las_size_ccsd = 1000
_max_las_size_ccsd_t = 1000
Regularized frag_wghtlist = [1. 1. 1. 1. 1. 1.]
    CPU time for LO and fragment        0.00 sec, wall time      0.00 sec

WARN: Input vhf is not found. Building vhf from SCF MO.

ao2mo est mem= 0.54 MB  avail mem= 3628.97 MB
aux blksize for incore ao2mo: 232/232
    CPU time for Integral xform         0.02 sec, wall time      0.00 sec
LO occ proj: 4 active | 1 standby | 3 orthogonal
LO vir proj: 2 active | 3 standby | 33 orthogonal
    CPU t

In [55]:
moliao.aoslice_by_atom()
aoslices = moliao.aoslice_by_atom()
for ia in range(moliao.natm):
    sh0, sh1, p0, p1 = aoslices[ia]
    symb = moliao.atom_symbol(ia)        # e.g. 'O', 'H1', 'H@2' (keeps any user labels)
    elem = moliao.atom_pure_symbol(ia)   # e.g. 'O', 'H'        (just the element)
    Z    = moliao.atom_charge(ia)        # nuclear charge
    xyz  = moliao.atom_coord(ia)         # coordinates in Bohr
    print(f"Atom {ia} ({symb}) {Z} {xyz} AOs {p0}:{p1}")

Atom 0 (O) 8 [-2.80655197 -0.21679801  0.        ] AOs 0:5
Atom 1 (H) 1 [-3.53079329  1.44053527  0.        ] AOs 5:6
Atom 2 (H) 1 [-1.00879882  0.07654796  0.        ] AOs 6:7
Atom 3 (O) 8 [2.67673782 0.21025931 0.        ] AOs 7:12
Atom 4 (H) 1 [ 3.29991847 -0.7066547  -1.43347254] AOs 12:13
Atom 5 (H) 1 [ 3.29991847 -0.7066547   1.43347254] AOs 13:14


In [62]:
mol.elements

['O', 'H', 'H', 'O', 'H', 'H']

In [37]:
from pyscf.tools import molden

with open('water_dimer_iao.molden', 'w') as f:
    molden.header(mol, f)
    molden.orbital_coeff(mol, f, lo_coeff)

In [3]:
from collections.abc import Iterable
from pyscf.lno import lnoccsd
from afqmc.lno_afqmc.lno_afqmc import lno_ccsd
from pyscf.lno.tools import autofrag_iao
from jax import random
import os

In [9]:
# from afqmc.lno_afqmc.lno_afqmc import prep_afqmc, run_lnoafqmc

def las_size(mf, frozen):
    mol = mf.mol
    nocc = np.count_nonzero(mf.mo_occ)
    actfrag = np.array([i for i in range(mol.nao) if i not in frozen])
    # frzocc = np.array([i for i in range(nocc) if i in frozen])
    actocc = np.array([i for i in range(nocc) if i in actfrag])
    actvir = np.array([i for i in range(nocc, mol.nao) if i in actfrag])
    # nfrzocc = len(frzocc)
    nactocc = len(actocc)
    nactvir = len(actvir)
    # nactorb = len(actfrag)
    return nactocc, nactvir

def ao_comp(mf, orbloc, ao_threshold=1e-2):
    mol = mf.mol
    S = mol.intor('int1e_ovlp')
    proj = (S @ orbloc)**2
    proj = proj / np.sum(proj, axis=0)
    if len(proj.shape) == 2:
        proj = np.sum(proj, axis=1)
    ao_labels = mol.ao_labels()
    above = np.where(proj > ao_threshold)[0]
    # sort them by contribution descending
    above = above[np.argsort(proj[above])[::-1]]
    ao_lines = []
    print(f"AOs with contribution > {ao_threshold}")
    ao_lines.append(f"AOs with contribution > {ao_threshold}")
    print(f"{'AO Label':>16s}  {'Amp':>6s}")
    ao_lines.append(f"{'AO Label':>16s}  {'Amp':>6s}")
    for idx in above:
        print(f"{ao_labels[idx]:>16s}  {proj[idx]:6.4f}")
        ao_lines.append(f"{ao_labels[idx]:>16s}  {proj[idx]:6.4f}") 
    ao_message = "\n".join(ao_lines)
    return ao_message, ao_labels[above[0]]

# def run_lnoccsd(
    #     mf, 
    #     options, # qmc parameters
    #     lo_coeff, 
    #     frag_lolist,
    #     nfrozen = 0, 
    #     thresh = 1e-6, 
    #     run_frg_list = None,
    #     atom_group = None, # = mol.elements if frag_lolist is grouped by atoms
    #     chol_cut = 1e-5,
    #     ):

    # mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=0)
    # mlno.lno_thresh = [thresh*10,thresh]
    # lno_thresh = mlno.lno_thresh
    # nfrag = len(frag_lolist)
    # lno_type = ['1h','1h']
    # lno_pct_occ = [None, None]
    # lno_norb = [[None,None]] * nfrag
    # eris = mlno.ao2mo()

    # # trial = options["trial"]

    # if run_frg_list is None:
    #     run_frg_list = range(nfrag)

    # run_frag_lolist = [frag_lolist[i] for i in run_frg_list]

    # print(f'Number of LNO-FRAGMENT: {nfrag}')

    # seeds = random.randint(random.PRNGKey(options["seed"]), shape=(nfrag,), minval=0, maxval=100*nfrag)
    # # options["max_error"] = options["max_error"]/np.sqrt(nfrag)
    # emp2 = np.zeros(nfrag, dtype='float64')
    # ecc  = np.zeros(nfrag, dtype='float64')
    # nact = np.zeros(nfrag, dtype='int32')

    # # Loop over fragment
    # for ifrag, loidx in enumerate(run_frag_lolist):
    #     print("\n")
    #     width = 80
    #     msg = f" RUNNING LNO-FRAGMENT {run_frg_list[ifrag]+1}/{nfrag} "
    #     print(msg.center(width, '='))
    #     if atom_group is not None:
    #         print(f"Center Atom {atom_group[ifrag]}")

    #     orbloc = lo_coeff[:,loidx]
    #     lno_param = [{'thresh': lno_thresh[i], 'pct_occ': lno_pct_occ[i],
    #                     'norb': lno_norb[ifrag][i]} for i in [0,1]]

    #     lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    #     nactocc, nactvir = las_size(mf, lno_frozen)
    #     print(f'number of active occupied orbitals: {nactocc}')
    #     print(f'number of active virtual orbitals: {nactvir}')

    #     # identify the center LO's AO component
    #     print(f'Locating Fragment {loidx} by AOs')
    #     ao_message = ao_comp(mf, orbloc)

    #     mo_occ = mlno.mo_occ
    #     lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, len(mo_occ))
    #     mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=3)
    #     mcc._s1e = mlno._s1e
    #     mcc._h1e = mlno._h1e
    #     mcc._vhf = mlno._vhf
    #     if mlno.kwargs_imp is not None:
    #         mcc = mcc.set(**mlno.kwargs_imp)
    #     # time0 = time.perf_counter()
    #     (eorb_mp2, eorb_cc), t1, t2 = \
    #         lno_ccsd(mcc, lno_coeff, uocc_loc, mo_occ, maskact) # <<< this is on CPU
    #     # time1 = time.perf_counter()

    #     print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    #     print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    #     prjlo = uocc_loc @ uocc_loc.T.conj()
    #     options["seed"] = seeds[ifrag]
    #     _, _ = prep_afqmc(mf, lno_coeff, t1, t2, lno_frozen, prjlo, options, chol_cut=chol_cut)
    #     run_lnoafqmc(options)
    #     os.system(f'mv afqmc.out lnoafqmc.out{run_frg_list[ifrag]+1}')

    #     emp2[ifrag] = eorb_mp2
    #     ecc[ifrag]  = eorb_cc
    #     nact[ifrag] = nactocc + nactvir

    # emp2_tot = sum(emp2)
    # ecc_tot = sum(ecc)
    # orbmax = max(nact)
    # print(f'LNO-MP2 Corr Energy: {eorb_mp2:.8f}')
    # print(f'LNO-CCSD Corr Energy: {eorb_cc:.8f}')
    # print(f'Largest LNO Fragment: {orbmax}')
    # return emp2_tot, ecc_tot, orbmax

In [12]:
ao_mess, ao1 = ao_comp(mf, mf.mo_coeff[:,3:5])

AOs with contribution > 0.01
        AO Label     Amp
      3 O 3pz     0.3324
      3 O 2pz     0.2835
      0 O 3s      0.2572
      0 O 2s      0.2207
      5 H 1s      0.1318
      4 H 1s      0.1318
      1 H 2s      0.1095
      2 H 2s      0.1028
      1 H 1s      0.0917
      2 H 1s      0.0869
      2 H 2px     0.0532
      1 H 2py     0.0406
      5 H 2s      0.0314
      4 H 2s      0.0314
      5 H 2py     0.0158
      4 H 2py     0.0158
      1 H 2px     0.0123


In [13]:
ao1

'3 O 3pz   '

In [ ]:
options = {'n_eql': 3,
           'n_prop_steps': 50,
           'n_ene_blocks': 1,
           'n_sr_blocks': 5,
           'n_blocks': 100,
           'n_walkers': 100,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'rhf',
           'trial': 'ccsd_pt2',
           'dt':0.005,
           'use_gpu': True,
           'max_error': 1e-3
           }

# import jax
# jax.config.update("jax_enable_x64", True)

mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_coeff_iao = lo.iao.iao(mol, orbocc)
lo_coeff_iao = lo.orth.vec_lowdin(lo_coeff_iao, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

frag_lolist = autofrag_iao(moliao)

In [7]:
run_lnoccsd(
        mf, 
        options = options,
        lo_coeff = lo_coeff_iao, 
        frag_lolist = frag_lolist,
        nfrozen = 0, 
        thresh = 3e-5, 
        run_frg_list = [0],
        atom_group = moliao.elements
        )

Number of LNO-FRAGMENT: 6


E0505 15:45:19.604932   26178 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0505 15:45:19.614143   25972 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)




=========================== RUNNING LNO-FRAGMENT 1/6 ===========================
Center Atom O
number of active occupied orbitals: 6
number of active virtual orbitals: 24
Locating Fragment [0, 1, 2, 3, 4] by AOs
AOs with contribution > 0.01
        AO Label     Amp
      0 O 1s      0.9019
      0 O 2px     0.5103
      0 O 2py     0.4829
      0 O 2pz     0.4539
      0 O 2s      0.4206
      0 O 3pz     0.4033
      0 O 3py     0.3156
      0 O 3s      0.3042
      0 O 3px     0.2529
      2 H 2px     0.2257
      1 H 2py     0.2102
      1 H 2px     0.0825
      1 H 1s      0.0811
      2 H 1s      0.0769
      1 H 2pz     0.0722
      2 H 2pz     0.0700
      2 H 2py     0.0626
      2 H 2s      0.0360
      1 H 2s      0.0332
E(MODIFIED_CCSD) = -152.2924460928207  E_corr = -0.229955445892751
LNO-MP2 Orbital Energy: -0.17177576
LNO-CCSD Orbital Energy: -0.17907859
# Calculating Effective Active Space One-electron Integrals
# number of active occupied orbitals 6
# number of active

E0505 15:45:22.493585   26460 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0505 15:45:22.501789   26392 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


Trial: ccsd_pt2(norb=30, nelec=(6, 6), n_opt_iter=30, n_batch=1)
Sampler: sampler_pt2(n_prop_steps=50, n_ene_blocks=1, n_sr_blocks=5, n_blocks=100, n_chol=148, n_micro_steps=10, n_macro_steps=5)
norb: 30
nelec: (6, 6)
nchol: 148
n_eql: 3
n_prop_steps: 50
n_ene_blocks: 1
n_sr_blocks: 5
n_blocks: 100
n_walkers: 100
n_batch: 1
seed: 189
walker_type: rhf
trial: ccsd_pt2
dt: 0.005
use_gpu: True
max_error: 0.001
n_exp_terms: 6
ene0: 0.0
Propagating with 100 walkers
Initial Orbital energy: -0.179067
----------------------- Equilibration -----------------------
Inv_T      E(Guide)   runTime
 0.00   -152.062489      4.37
12.50   -152.300146      8.24 
25.00   -152.293297     11.79 
37.50   -152.300306     14.08 
--------------------- Sampling Blocks -----------------------
Target Final Error ~ 0.001000
   N      E(Guide)     Error      E(Orb)     Error  Kill_WK      Time
  10   -152.279415  0.004649   -0.181886  0.000482        0     19.18
  20   -152.290320  0.004686   -0.182051  0.000417     

(np.float64(-0.17177576269707853),
 np.float64(-0.17907858640613306),
 np.int32(30))